# Mental Health RAG Chatbot (Mistral-7B Edition)

Run the cell below to install the necessary libraries for Mistral logic.

In [2]:
%pip install chromadb sentence-transformers huggingface_hub pandas gradio python-dotenv torch



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    

In [3]:
import pandas as pd
import json

# Path to the log we created in the main script
LOG_PATH = "rag_logs.jsonl"

def view_rag_traceability(limit=10):
    if not os.path.exists(LOG_PATH):
        print(f"⚠️ Log file not found at {LOG_PATH}. Have you started chatting yet?")
        return
    
    data = []
    with open(LOG_PATH, "r") as f:
        for line in f:
            entry = json.loads(line)
            # Extract basic info
            msg = entry.get("user_message", "")
            intent = entry.get("intent", "")
            
            # Extract details about the dataset reference
            refs = entry.get("retrieved_sources", [])
            for ref in refs:
                data.append({
                    "Time": entry["timestamp"].split("T")[1][:8], # HH:MM:SS
                    "User Says": msg,
                    "Intent": intent,
                    "Source Dataset": ref["collection"],
                    "Row ID": ref["doc_id"],
                    "Dataset Text (What AI used)": ref["text_snippet"]
                })

    df = pd.DataFrame(data)
    # Display the most recent logs first
    return df.tail(limit).iloc[::-1]

# Run it!
view_rag_traceability(limit=5)


⚠️ Log file not found at rag_logs.jsonl. Have you started chatting yet?


: 

In [11]:
import os, json, time, chromadb, gradio as gr
from huggingface_hub import InferenceClient
from chromadb.utils import embedding_functions
from pathlib import Path

# ══════════════════════════════════════════════════════════
# 1. CONFIG & SETUP
# ══════════════════════════════════════════════════════════
HF_TOKEN     = os.getenv("HUGGINGFACE_TOKEN")
MODEL_NAME   = "Qwen/Qwen2.5-7B-Instruct"
DB_PATH      = "./chroma_db"
HISTORY_FILE = "chat_history.jsonl"
PROFILE_FILE = "user_profile.json"

# Function using the 'time' module as requested
def get_time_context():
    hour = time.localtime().tm_hour
    if 5 <= hour < 12: return "morning"
    elif 12 <= hour < 17: return "afternoon"
    elif 17 <= hour < 22: return "evening"
    else: return "night"

def classify_intent(text):
    t = text.lower()
    if any(p in t for p in ["kill myself","suicide","self harm"]): return "high_risk"
    if any(p in t for p in ["hi","hello","hey","kashi","kaisi"]): return "greeting"
    return "general"

# ══════════════════════════════════════════════════════════
# 2. DATA PERSISTENCE (PROFILES & HISTORY)
# ══════════════════════════════════════════════════════════
def load_history():
    pairs = []
    if Path(HISTORY_FILE).exists():
        for line in open(HISTORY_FILE):
            e = json.loads(line)
            pairs.append([e["user"], e["bot"]])
    return pairs

def save_message(user, bot):
    with open(HISTORY_FILE, "a") as f:
        f.write(json.dumps({"user": user, "bot": bot}) + "\n")

def load_profile():
    if Path(PROFILE_FILE).exists(): return json.load(open(PROFILE_FILE))
    return {"name": None, "study_field": None, "struggles": []}

# ══════════════════════════════════════════════════════════
# 3. CHAT LOGIC (MyAlly Personality)
# ══════════════════════════════════════════════════════════
client = InferenceClient(model=MODEL_NAME, token=HF_TOKEN)

# Initialize database
try:
    chroma_client = chromadb.PersistentClient(path=DB_PATH)
    ef = embedding_functions.SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")
    empathy_col = chroma_client.get_collection("student_support_empathy", embedding_function=ef)
except:
    empathy_col = None

def chat_logic(user_message, history):
    time_ctx = get_time_context()
    intent = classify_intent(user_message)
    profile = load_profile()

    # Safety trigger
    if intent == "high_risk":
        return "I’m really glad you told me 💙 Please reach out to a trusted person or a helpline right now. You are not alone."

    # RAG Context
    context = ""
    if empathy_col:
        try:
            res = empathy_col.query(query_texts=[user_message], n_results=2)
            context = "\n".join(res["documents"][0])
        except: pass

    # 🧠 SYSTEM PROMPT (Hinglish/Minglish & Human Vibe)
    system_prompt = f"""
You are MyAlly — a warm, genuine, and "chill" Indian friend.
- LANGUAGE: Always match the user. If they use Hinglish or Minglish, you MUST reply in that same mixed style using Roman script.
- CHARACTER: Talk like a real person, not a support bot. Use words like 'mast', 'bhari', 'sahi hai', 'vibe'.
- NO PARROTING: Never repeat the user's sentence back to them. 
- CONTEXT: It's {time_ctx} right now. Match the {time_ctx} energy.
"""

    messages = [{"role": "system", "content": system_prompt}]
    
    # Add recent history for memory
    for h in history[-4:]:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})

    messages.append({
        "role": "user", 
        "content": f"Inspiration: {context}\n\nUser: {user_message}\n\nReply naturally in their style:"
    })

    try:
        # Temperature 0.8 keeps conversation fresh and spontaneous
        res = client.chat_completion(messages=messages, max_tokens=350, temperature=0.8)
        reply = res.choices[0].message.content
    except Exception as e:
        reply = f"Error connecting to model: {e}"

    save_message(user_message, reply)
    return reply

# ══════════════════════════════════════════════════════════
# 4. LAUNCH UI
# ══════════════════════════════════════════════════════════
demo = gr.ChatInterface(
    fn=chat_logic,
    chatbot=gr.Chatbot(value=load_history()),
    title="MyAlly 💙",
    description="Your mental health friend. Natural, human, and time-aware."
)

if __name__ == "__main__":
    demo.launch(debug=True)


